# Processing + Gridding SPC Hail Reports and Creating CSV Files

### Importing Packages

In [1]:
from pathlib import Path
import xarray as xr
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from pyproj import Transformer

### Opening and Preprocessing SPC Hail Reports

Download the SPC Hail Reports here:

https://niuits-my.sharepoint.com/:x:/g/personal/z1953922_students_niu_edu/IQAhMlIexo1VSoFBusD8ILm-AVgU4IvaSDGZCUG0PD3iih8?e=7cIXcc

In [2]:
hail_df = pd.read_csv('1955-2024_hail.csv')

hail_df = hail_df[['date', 'time', 'slat', 'slon', 'mag']].dropna()

hail_df['datetime'] = pd.to_datetime(hail_df['date'].astype(str) + ' ' + hail_df['time'].astype(str), errors='coerce')

hail_df = hail_df[(hail_df['datetime'].dt.year >= 2000) & (hail_df['datetime'].dt.year <= 2019)
                  & (hail_df['datetime'].dt.month == 5) & (hail_df['mag'] >= 1)]
hail_df

,date,time,slat,slon,mag,datetime
122139,2000-05-10,00:17:00,43.0800,-76.8800,1.00,2000-05-10 00:17:00
122141,2000-05-10,00:40:00,40.3800,-80.4000,1.00,2000-05-10 00:40:00
122146,2000-05-10,05:25:00,42.7300,-73.5700,1.00,2000-05-10 05:25:00
122148,2000-05-10,05:30:00,41.7000,-74.5800,1.00,2000-05-10 05:30:00
122149,2000-05-10,05:35:00,42.0700,-73.9200,1.75,2000-05-10 05:35:00
...,...,...,...,...,...,...
358043,2019-05-09,22:30:00,31.5845,-94.6795,1.50,2019-05-09 22:30:00
358044,2019-05-09,22:40:00,29.0400,-95.8800,1.50,2019-05-09 22:40:00
358045,2019-05-09,22:54:00,30.1800,-93.3700,3.00,2019-05-09 22:54:00
358046,2019-05-09,23:09:00,30.2270,-93.3590,2.50,2019-05-09 23:09:00


### Gridding and Hail Class Functions

In [ ]:
# This function assigns the maximum hail size from SPC Storm Reports to each GEFS grid point within an 80-km radius
def grid_hail_to_gefs(hail_df, grid_lats, grid_lons, start_time, end_time, radius_km=80):

    df = hail_df[(hail_df['datetime'] >= start_time) & (hail_df['datetime'] < end_time)]

    if len(df) == 0:
        return np.zeros_like(grid_lats)

    # Go over to meters
    transformer = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)
    x_reports, y_reports = transformer.transform(df['slon'].values, df['slat'].values)

    # Make the grid and cKDTree (used AI help me out with this part)
    x_grid, y_grid = transformer.transform(grid_lons.flatten(), grid_lats.flatten())
    tree = cKDTree(np.column_stack([x_reports, y_reports]))

    # 80000 meters
    radius_m = radius_km * 1000

    neighbors = tree.query_ball_point(np.column_stack([x_grid, y_grid]), r=radius_m)

    hail_flat = np.zeros(len(x_grid))
    hail_sizes = df['mag'].values

    for i, inds in enumerate(neighbors):
        if len(inds) > 0:
            hail_flat[i] = np.max(hail_sizes[inds])

    return hail_flat.reshape(grid_lats.shape)

# Let's put hail sizes into classes (this was moreso for the original project idea)
def bin_hail(hail_grid):
    bins = [0, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 10]

    return np.digitize(hail_grid, bins) - 1

### Making Yearly CSV Files with Gridded SPC Hail, 24-hr Max SBCAPE, and 24-hr Max 850-250 hPa Wind Shear

(This was done on TRITON with large data)

In [ ]:
out_dir = Path('/home/scratch/lmoeller/GEFSv12_ref_483/GEFSv12_ref_hail_csv')
out_dir.mkdir(parents=True, exist_ok=True)

# Goal is to combine the CAPE and shear data with the hail reports
base_path = Path('/home/scratch/lmoeller/GEFSv12_ref_483/cape_shear_nc')

# Looping through all 20 years
for year in range(2000, 2020):

    year_rows = []

    # May only
    for month in [5]:

        for day in range(1, 32):

            file_path = base_path / f"{year}" / f"{month:02d}" / \
                f'gefs_c00_{year}{month:02d}{day:02d}00_cape_shear_subset.nc'

            ds = xr.open_dataset(file_path)

            lats_1d = ds['latitude'].values
            lons_1d = ds['longitude'].values
            lons, lats = np.meshgrid(lons_1d, lats_1d)

            init_time = pd.Timestamp(f'{year}-{month:02d}-{day:02d} 00:00')
            valid_time = ds['valid_time'].values

            # Days 1-5 Forecast
            for fday in range(1, 6):

                start = init_time + pd.Timedelta(hours=24 * (fday - 1))
                end   = init_time + pd.Timedelta(hours=24 * fday)

                mask_gefs = (valid_time >= np.datetime64(start)) & \
                            (valid_time < np.datetime64(end))

                cape = ds['cape'].sel(step=mask_gefs).max(dim='step')
                shear = ds['shear_850_250'].sel(step=mask_gefs).max(dim='step')

                cape_grid = cape.values
                shear_grid = shear.values

                # Calling the gridding and binning function
                hail_grid = grid_hail_to_gefs(hail_df, lats, lons, start, end)
                hail_classes = bin_hail(hail_grid)

                df_out = pd.DataFrame({'year': year, 'init_date': str(init_time), 'lead_time': fday, 'lat': lats.flatten(),
                    'lon': lons.flatten(), 'cape': cape_grid.flatten(), 'shear': shear_grid.flatten(), 'hail_class': hail_classes.flatten()})

                df_out = df_out.dropna()
                year_rows.append(df_out)

            ds.close()

    if len(year_rows) > 0:

        year_df = pd.concat(year_rows, ignore_index=True)

        year_df['cape'] = year_df['cape'].astype('float32')
        year_df['shear'] = year_df['shear'].astype('float32')
        year_df['lat'] = year_df['lat'].astype('float32')
        year_df['lon'] = year_df['lon'].astype('float32')
        year_df['hail_class'] = year_df['hail_class'].astype('int8')

        out_file = out_dir / f'GEFSv12_ref_hail_{year}.csv'
        year_df.to_csv(out_file, index=False)
        print(f'Saved: {out_file}')

### Let's See What the CSV File Looks Like

Download the 2000 csv file from the OneDrive to run this:

https://niuits-my.sharepoint.com/:x:/g/personal/z1953922_students_niu_edu/IQB3-NLueJMaQ4_EHEdIU_5xAQxb9q5c99237M7qjaU-0jw?e=NouzfD

In [3]:
df = pd.read_csv('GEFSv12_ref_hail_2000.csv')
df

,year,init_date,lead_time,lat,lon,cape,shear,hail_class
0,2000,2000-05-01 00:00:00,1,51.00,-107.00,0.0,35.231506,0.0
1,2000,2000-05-01 00:00:00,1,51.00,-106.75,0.0,36.458935,0.0
2,2000,2000-05-01 00:00:00,1,51.00,-106.50,0.0,37.659780,0.0
3,2000,2000-05-01 00:00:00,1,51.00,-106.25,0.0,38.893764,0.0
4,2000,2000-05-01 00:00:00,1,51.00,-106.00,0.0,40.016552,0.0
...,...,...,...,...,...,...,...,...
946883,2000,2000-05-10 00:00:00,5,34.75,-70.75,0.0,14.221797,0.0
946884,2000,2000-05-10 00:00:00,5,34.75,-70.50,0.0,15.117097,0.0
946885,2000,2000-05-10 00:00:00,5,34.75,-70.25,0.0,15.181764,0.0
946886,2000,2000-05-10 00:00:00,5,34.75,-70.00,0.0,14.805243,0.0


In [6]:
print(df['hail_class'].value_counts())
df.describe()

hail_class
0.0    929176
2.0      7713
1.0      7290
4.0      1636
3.0       471
8.0       295
5.0       249
6.0        57
Name: count, dtype: int64


,year,lead_time,lat,lon,cape,shear,hail_class
count,946888.0,946888.000000,946888.000000,946888.000000,946887.000000,946887.000000,946887.000000
mean,2000.0,2.983183,37.069251,-86.000376,762.717887,28.666460,0.036562
std,0.0,1.408156,8.147818,12.196149,1004.770026,15.328720,0.315826
min,2000.0,1.000000,23.000000,-107.000000,0.000000,1.431089,0.000000
25%,2000.0,2.000000,30.000000,-96.500000,11.000000,16.532906,0.000000
50%,2000.0,3.000000,37.000000,-86.000000,245.000000,24.790539,0.000000
75%,2000.0,4.000000,44.000000,-75.500000,1225.000000,39.698469,0.000000
max,2000.0,5.000000,51.000000,-65.000000,5871.000000,87.781290,8.000000
